In [1]:
import os
import subprocess
import time
import shutil
import csv
from datetime import datetime
import stat

import ast
import json
import shutil
import builtins
import re
import pandas as pd

import cProfile
import pstats
import tempfile

from IPython.display import display

import timeit
import contextlib
import io
import threading


In [2]:
# KONFIGURASI

input_file = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\asli\nim_github(10).txt"
projects_folder = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(10)"
os.makedirs(projects_folder, exist_ok=True)

In [3]:
#LOG Folder/output folder

log_folder = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\OutputPreprocessing"
os.makedirs(log_folder, exist_ok=True)

### konfigurasi log proses clone

In [21]:
# konfigurasi output untuk proses clone
result_excel = os.path.join(log_folder, "hasil_clone.xlsx")
result_json = os.path.join(log_folder, "hasil_clone.json")

In [22]:
# LOG CLONE
log_file = os.path.join(log_folder, "log_clone.csv")

# SETUP LOG

if not os.path.exists(log_file):
    with open(log_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "nim", "url", "status", "message"])

def write_log(nim, url, status, message):
    with open(log_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            nim,
            url,
            status,
            message
        ])

# Pengumpulan Data

In [23]:
# BACA DATASET
# dataset berupa nim | url
def load_dataset(file_path):

    dataset = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split("|")

            if len(parts) != 2:
                continue

            nim = parts[0].strip()
            url = parts[1].strip()

            dataset.append((nim, url))

    return dataset

In [24]:
# HANYA SIMPAN .py & .ipynb

def keep_only_code_files(repo_path):

    for root, dirs, files in os.walk(repo_path):

        # Jangan masuk folder .git
        if ".git" in dirs:
            dirs.remove(".git")

        for file in files:
            if not (file.endswith(".py") or file.endswith(".ipynb")):
                try:
                    os.remove(os.path.join(root, file))
                except:
                    pass

    # Hapus folder kosong
    for root, dirs, files in os.walk(repo_path, topdown=False):
        if not os.listdir(root):
            try:
                os.rmdir(root)
            except:
                pass


In [25]:
# HITUNG FILE KODE

def count_code_files(repo_path):

    py_count = 0
    ipynb_count = 0

    for root, dirs, files in os.walk(repo_path):

        if ".git" in dirs:
            dirs.remove(".git")

        for file in files:
            if file.endswith(".py"):
                py_count += 1
            elif file.endswith(".ipynb"):
                ipynb_count += 1

    return py_count, ipynb_count

In [26]:
# Remove folder gagal

def remove_readonly(func, path, exc_info):
    """
    Menghapus atribut read-only lalu retry delete.
    """
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print(f"Gagal paksa hapus: {path}")
        print(f"->   Alasan: {str(e)}")

In [27]:
# CLONE FUNCTION

results = []

def clone_repo(nim, url):

    global success_count, fail_count, skip_count
    global total_py_files, total_ipynb_files

    # Jika NIM kosong
    if not nim:
        print(f"[SKIP] URL tanpa NIM → {url}")
        write_log("", url, "SKIPPED", "NIM kosong")
        skip_count += 1

        results.append({
            "nim": nim,
            "url": url,
            "status": "SKIPPED",
            "message" : "NIM Kosong",
            "py_files": 0,
            "ipynb_files": 0,
            "total_files": 0
        })
        return

    # Bersihkan URL jika ada /tree/
    if "/tree/" in url:
        url = url.split("/tree/")[0]

    target_path = os.path.join(projects_folder, nim)

    # Jika sudah pernah clone
    if os.path.exists(target_path):
        print(f"[SKIP] {nim} sudah ada.")
        write_log(nim, url, "SKIPPED", "Folder sudah ada")
        skip_count += 1

        results.append({
            "nim": nim,
            "url": url,
            "status": "SKIPPED",
            "message" : "Folder sudah ada",
            "py_files": 0,
            "ipynb_files": 0,
            "total_files": 0
        })

        return

    print(f"[CLONE] {url}")

    try:
        subprocess.run(
            ["git", "clone", url, target_path],
            check=True,
            timeout=300,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE
        )

        # Simpan hanya .py dan .ipynb
        keep_only_code_files(target_path)

        # Hitung jumlah file kode
        py_count, ipynb_count = count_code_files(target_path)

        total_py_files += py_count
        total_ipynb_files += ipynb_count

        print(f"   ✅ Berhasil clone {nim}  ||      Jumlah file dalam repo .py: {py_count} | .ipynb: {ipynb_count}")

        write_log(nim, url, "SUCCESS", f"Clone berhasil | py:{py_count} ipynb:{ipynb_count}")

        success_count += 1

        results.append({
            "nim": nim,
            "url": url,
            "status": "SUCCESS",
            "message" : "Clone berhasil",
            "py_files": py_count,
            "ipynb_files": ipynb_count,
            "total_files": py_count + ipynb_count
        })

        time.sleep(1)

    except subprocess.CalledProcessError as e:

        error_msg = e.stderr.decode(errors="ignore")

        print(f"   ❌ Gagal clone {nim}")
        print(f"   Alasan:\n{error_msg}")

        write_log(nim, url, "FAILED", error_msg)

        # Hapus folder jika setengah clone
        if os.path.exists(target_path):
            try:
                shutil.rmtree(target_path, onerror=remove_readonly)
                print("   🧹 Folder clone dihapus")
            except Exception as delete_error:
                print("   ⚠ Gagal hapus folder clone")
                print(str(delete_error))

        fail_count += 1

        results.append({
            "nim": nim,
            "url": url,
            "status": "FAILED",
            "message" : error_msg,
            "py_files": 0,
            "ipynb_files": 0,
            "total_files": 0
        })

    except subprocess.TimeoutExpired:

        print(f"   ⏰ Timeout clone {nim}")
        write_log(nim, url, "FAILED", "Timeout")

        if os.path.exists(target_path):
            try:
                shutil.rmtree(target_path)
            except:
                pass

        fail_count += 1

        results.append({
            "nim": nim,
            "url": url,
            "status": "FAILED",
            "message" : "Clone Timeout",
            "py_files": 0,
            "ipynb_files": 0,
            "total_files": 0
        })

In [28]:
dataset = load_dataset(input_file)

total_url = len(dataset)
success_count = 0
fail_count = 0
skip_count = 0
total_py_files = 0
total_ipynb_files = 0

print(f"\nTotal URL dalam dataset: {total_url}\n")

for nim, url in dataset:
    clone_repo(nim, url)

print("\n===== RINGKASAN CLONING =====")
print(f"Total URL      : {total_url}")
print(f"Berhasil clone : {success_count}")
print(f"Gagal clone    : {fail_count}")
print(f"Skipped        : {skip_count}")
print(f"Total file .py    : {total_py_files}")
print(f"Total file .ipynb : {total_ipynb_files}")
print(f"Total file kode   : {total_py_files + total_ipynb_files}")
print("================================")
print(f"Log tersimpan di: {log_file}")


Total URL dalam dataset: 10

[CLONE] https://github.com/Katakon17/2241720092_ML_2025
   ✅ Berhasil clone 2241720092  ||      Jumlah file dalam repo .py: 0 | .ipynb: 22
[CLONE] https://github.com/4rdnac/2341720187_ML_2025
   ✅ Berhasil clone 2341720187  ||      Jumlah file dalam repo .py: 1 | .ipynb: 51
[CLONE] https://github.com/fajrulsantoso/244107023010_ML_2025
   ❌ Gagal clone 244107023010
   Alasan:
Cloning into 'D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(10)\244107023010'...
error: invalid path 'JS08 /JS08'
fatal: unable to checkout working tree
You can inspect what was checked out with 'git status'
and retry with 'git restore --source=HEAD :/'


   🧹 Folder clone dihapus
[CLONE] https://github.com/FarrelAD/2341720081_ML_2025
   ✅ Berhasil clone 2341720081  ||      Jumlah file dalam repo .py: 0 | .ipynb: 52
[CLONE] https://github.com/emkafie/Machine-Learning
   ✅ Berhasil clone 2341720176  ||      Jumlah file dalam repo .py: 0 | .ipynb: 51
[CLONE] https://github.com/Ast

In [29]:
# hasil clone 
df = pd.DataFrame(results)

# simpan file hasil
df.to_excel(result_excel, index=False)

with open(result_json, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

print("\nFile hasil disimpan di:")
print(result_excel)
print(result_json)

# TAMPILKAN TABEL
print("\n===== DATA HASIL CLONING =====\n")
display(df)


File hasil disimpan di:
D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\OutputPreprocessing\hasil_clone.xlsx
D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\OutputPreprocessing\hasil_clone.json

===== DATA HASIL CLONING =====



,nim,url,status,message,py_files,ipynb_files,total_files
0,2241720092,https://github.com/Katakon17/2241720092_ML_2025,SUCCESS,Clone berhasil,0,22,22
1,2341720187,https://github.com/4rdnac/2341720187_ML_2025,SUCCESS,Clone berhasil,1,51,52
2,244107023010,https://github.com/fajrulsantoso/244107023010_...,FAILED,Cloning into 'D:\PUTRI\D4\SEMESTER 8 (4C)\skri...,0,0,0
3,2341720081,https://github.com/FarrelAD/2341720081_ML_2025,SUCCESS,Clone berhasil,0,52,52
4,2341720176,https://github.com/emkafie/Machine-Learning,SUCCESS,Clone berhasil,0,51,51
5,2341720095,https://github.com/AstorBoy11/2341720095_ML_20...,SUCCESS,Clone berhasil,0,43,43
6,2341720217,https://github.com/NathanaelGracedo/2341720217...,FAILED,Cloning into 'D:\PUTRI\D4\SEMESTER 8 (4C)\skri...,0,0,0
7,2341720117,https://github.com/Oktavian19/2341720117_ML_2025,SUCCESS,Clone berhasil,9,50,59
8,2341720096,https://github.com/jioooo20/2341720096_ML_2025,SUCCESS,Clone berhasil,1,54,55
9,2341720163,https://github.com/SatrioHubs/2341720163_ML_2025,SUCCESS,Clone berhasil,0,4,4


# convert ke py

In [31]:
# konfigurasi folder hasil convert
target_folder = projects_folder + "_py"
os.makedirs(target_folder, exist_ok=True)

In [32]:
#menghapus magic command dari ipynb
def clean_magic_commands(code):

    code = re.sub(r"^\s*%.*$", "", code, flags=re.MULTILINE)
    code = re.sub(r"^\s*!.*$", "", code, flags=re.MULTILINE)
    return code

In [33]:
# KONVERSI NOTEBOOK

def convert_ipynb_to_py(ipynb_path, py_path):

    try:

        with open(ipynb_path, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        code_cells = []

        for cell in notebook.get("cells", []):

            if cell.get("cell_type") != "code":
                continue

            source = "".join(cell.get("source", []))
            source = clean_magic_commands(source)

            code_cells.append(source)

        if not code_cells:
            raise Exception("Notebook tidak memiliki code cell")

        code = "\n\n".join(code_cells)

        with open(py_path, "w", encoding="utf-8") as f:
            f.write(code)

        return True, None

    except Exception as e:
        return False, str(e)


In [36]:
total_py_original = 0
total_ipynb = 0
converted_success = 0
converted_failed = 0

for root, dirs, files in os.walk(projects_folder):

    relative_path = os.path.relpath(root, projects_folder)
    new_root = os.path.join(target_folder, relative_path)

    os.makedirs(new_root, exist_ok=True)

    for file in files:

        source_path = os.path.join(root, file)

        # FILE PY ASLI

        if file.endswith(".py"):

            target_path = os.path.join(new_root, file)

            shutil.copy2(source_path, target_path)

            total_py_original += 1

        # FILE IPYNB

        elif file.endswith(".ipynb"):

            total_ipynb += 1

            new_name = file.replace(".ipynb", ".py")
            target_path = os.path.join(new_root, new_name)

            success, error = convert_ipynb_to_py(source_path, target_path)

            if success:

                converted_success += 1
                print(f"✔ Converted: {source_path}")

            else:

                converted_failed += 1
                print(f"❌ Gagal convert: {source_path}")
                print(f"   Alasan: {error}")

# STATISTIK AKHIR

total_py_output = total_py_original + converted_success
total_files_dataset = total_py_original + total_ipynb

print("\n===== STATISTIK KONVERSI DATASET =====")

print(f"Total file .py asli        : {total_py_original}")
print(f"Total file .ipynb asli     : {total_ipynb}")

print(f"\nBerhasil convert         : {converted_success}")
print(f"Gagal convert              : {converted_failed}")

print(f"\nTotal file .py output    : {total_py_output}")

print("\nDataset hasil convert tersimpan di:")
print(target_folder)

✔ Converted: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(10)\2241720092\Uts.ipynb
✔ Converted: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(10)\2241720092\Artificial Neural Network (ANN) dan Evaluasi Classifier\P1_JS13.ipynb
✔ Converted: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(10)\2241720092\Artificial Neural Network (ANN) dan Evaluasi Classifier\P2_JS13.ipynb
✔ Converted: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(10)\2241720092\Artificial Neural Network (ANN) dan Evaluasi Classifier\P3_JS13.ipynb
✔ Converted: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(10)\2241720092\Artificial Neural Network (ANN) dan Evaluasi Classifier\TP_JS13.ipynb
✔ Converted: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(10)\2241720092\Convolutional Neural Network\P1_JS14.ipynb
✔ Converted: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(10)\2241720092\Convolutional Neural Network\P2_JS14.ipynb
✔ Converted: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kod

# Waktu eksekusi

## seluruh project (CProfile)

In [47]:
# KONFIGURASI

dataset_folder = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(10)_py"

output_csv = os.path.join(log_folder, "runtime_cprofile.csv")
output_excel = os.path.join(log_folder, "runtime_cprofile.xlsx")
output_json = os.path.join(log_folder, "runtime_cprofile.json")

TIMEOUT = 30

In [48]:
# FUNCTION PROFILING
def run_with_cprofile(file_path):

    try:

        # membuat file sementara untuk profil
        with tempfile.NamedTemporaryFile(delete=False) as temp_prof:
            profile_file = temp_prof.name

        subprocess.run(
            [
                "python",
                "-m",
                "cProfile",
                "-o",
                profile_file,
                file_path
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
            timeout=TIMEOUT
        )

        stats = pstats.Stats(profile_file)

        runtime = stats.total_tt

        os.remove(profile_file)

        return "SUCCESS", runtime, ""

    except subprocess.TimeoutExpired:

        return "TIMEOUT", None, "Execution timeout"

    except Exception as e:

        return "FAILED", None, str(e)

In [49]:
# EKSTRAK METADATA

def extract_metadata(file_path, dataset_folder):

    # ambil path relatif dari dataset
    rel_path = os.path.relpath(file_path, dataset_folder)

    parts = rel_path.split(os.sep)

    # file name
    file_name = parts[-1]

    # nim = folder pertama
    nim = parts[0] if len(parts) >= 2 else "unknown"

    # module = folder setelah nim (jika ada)
    module = parts[1] if len(parts) >= 3 else "root"

    return nim, module, file_name

In [50]:
# LOOP DATASET

records = []
total_files = 0

print("\n========== MULAI PROFILING ==========\n")

for root, dirs, files in os.walk(dataset_folder):

    for file in files:

        if not file.endswith(".py"):
            continue

        file_path = os.path.join(root, file)

        nim, module, file_name = extract_metadata(file_path, dataset_folder)

        print(f"[RUN] {nim}/{module}/{file_name}")

        status, runtime, message = run_with_cprofile(file_path)

        total_files += 1

        records.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "nim": nim,
            "module": module,
            "file_name": file_name,
            "file_path": file_path,
            "status": status,
            "execution_time_seconds": runtime,
            "message": message
        })

print("\n========== PROFILING SELESAI ==========\n")


========== MULAI PROFILING ==========

[RUN] 2241720092/root/Uts.py
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P2_JS13.py
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P3_JS13.py
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/TP_JS13.py
[RUN] 2241720092/Convolutional Neural Network/P1_JS14.py
[RUN] 2241720092/Convolutional Neural Network/P2_JS14.py
[RUN] 2241720092/Convolutional Neural Network/TP_JS14.py
[RUN] 2241720092/Ekstraksi Fitur/P1_JS03.py
[RUN] 2241720092/Ekstraksi Fitur/P2_JS03.py
[RUN] 2241720092/Ekstraksi Fitur/P3_JS03.py
[RUN] 2241720092/Ekstraksi Fitur/P4_JS03.py
[RUN] 2241720092/Ekstraksi Fitur/TP_JS03.py
[RUN] 2241720092/Klasterisasi/P1_JS04.py
[RUN] 2241720092/Klasterisasi/P2_JS04.py
[RUN] 2241720092/Klasterisasi/P3_JS04.py
[RUN] 2241720092/Klasterisasi/TP_JS04.py
[RUN] 2241720092/Pemahaman data/P1_JS02.

In [51]:
# DATAFRAME
df = pd.DataFrame(records)

df["execution_time_seconds"] = df["execution_time_seconds"].round(6)

# SIMPAN CSV
df.to_csv(output_csv, index=False)
df.to_excel(output_excel, index=False)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=4)

# TAMPILKAN TABEL
print("========== TABEL HASIL RUNTIME ==========\n")

display_columns = [
    "nim",
    "module",
    "file_name",
    "status",
    "execution_time_seconds"
]

display(df[display_columns])

========== TABEL HASIL RUNTIME ==========



,nim,module,file_name,status,execution_time_seconds
0,2241720092,root,Uts.py,SUCCESS,0.481802
1,2241720092,Artificial Neural Network (ANN) dan Evaluasi C...,P1_JS13.py,SUCCESS,4.359169
2,2241720092,Artificial Neural Network (ANN) dan Evaluasi C...,P2_JS13.py,SUCCESS,3.465349
3,2241720092,Artificial Neural Network (ANN) dan Evaluasi C...,P3_JS13.py,SUCCESS,0.313593
4,2241720092,Artificial Neural Network (ANN) dan Evaluasi C...,TP_JS13.py,SUCCESS,0.001052
...,...,...,...,...,...
332,2341720187,JS15,P1_JS15.py,SUCCESS,1.329856
333,2341720187,JS15,P2_JS15.py,SUCCESS,0.216342
334,2341720187,JS15,app.py,SUCCESS,0.000892
335,2341720187,UAS,UAS.py,SUCCESS,0.001929


In [52]:
# STATISTIK RINGKAS

print("\n========== RINGKASAN ==========\n")

summary = {
    "Total file dijalankan": total_files,
    "SUCCESS": (df["status"] == "SUCCESS").sum(),
    "FAILED": (df["status"] == "FAILED").sum(),
    "TIMEOUT": (df["status"] == "TIMEOUT").sum(),
    "Rata-rata runtime": round(df["execution_time_seconds"].mean(), 6)
}

summary_df = pd.DataFrame(list(summary.items()), columns=["Metric", "Value"])

display(summary_df)

print("\nTop 5 runtime terlama:")

top_runtime = (
    df.sort_values("execution_time_seconds", ascending=False)
    [["nim","module","file_name","execution_time_seconds"]]
    .head()
)

display(top_runtime)

print("\nHasil lengkap disimpan di:")
print(output_csv)
print(output_excel)
print(output_json)


========== RINGKASAN ==========



,Metric,Value
0,Total file dijalankan,337.000000
1,SUCCESS,333.000000
2,FAILED,3.000000
3,TIMEOUT,1.000000
4,Rata-rata runtime,0.694314



Top 5 runtime terlama:


,nim,module,file_name,execution_time_seconds
1,2241720092,Artificial Neural Network (ANN) dan Evaluasi C...,P1_JS13.py,4.359169
138,2341720096,JS06,teoridasar.py,3.745389
276,2341720176,2341720176_ML_2025,P1_JS13.py,3.722280
2,2241720092,Artificial Neural Network (ANN) dan Evaluasi C...,P2_JS13.py,3.465349
58,2341720081,JS11,P4_JS11.py,3.208594



Hasil lengkap disimpan di:
D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\OutputPreprocessing\runtime_cprofile.csv
D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\OutputPreprocessing\runtime_cprofile.xlsx
D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\OutputPreprocessing\runtime_cprofile.json


## per function (timeit)

In [5]:
# KONFIGURASI
dataset_folder = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(10)_py"

output_csv = os.path.join(log_folder, "runtime_per_function2.csv")
output_excel = os.path.join(log_folder, "runtime_per_function2.xlsx")
output_json = os.path.join(log_folder, "runtime_per_function2.json")

RUN_REPEAT = 3
TIMEOUT = 30

In [6]:
# BLOK INPUT
builtins.input = lambda *args: "0"

In [7]:
# EKSTRAK METADATA

def extract_metadata(file_path, dataset_folder):

    # ambil path relatif dari dataset
    rel_path = os.path.relpath(file_path, dataset_folder)

    parts = rel_path.split(os.sep)

    # file name
    file_name = parts[-1]

    # nim = folder pertama
    nim = parts[0] if len(parts) >= 2 else "unknown"

    # module = folder setelah nim (jika ada)
    module = parts[1] if len(parts) >= 3 else "root"

    return nim, module, file_name

In [8]:
# EXTRACT FUNCTION DARI AST
def extract_functions(file_path):

    with open(file_path, "r", encoding="utf8", errors="ignore") as f:
        code = f.read()

    try:
        tree = ast.parse(code)
    except:
        return []

    functions = []

    for node in tree.body:

        if isinstance(node, ast.FunctionDef):

            func_code = ast.get_source_segment(code, node)

            functions.append((node.name, func_code))

    return functions

In [9]:
# GENERATE WRAPPER
def generate_wrapper(func_name, func_code):

    try:

        tree = ast.parse(func_code)
        func_node = tree.body[0]

        param_count = len(func_node.args.args)

        dummy_args = ",".join(["0"] * param_count)

    except:

        dummy_args = ""

    wrapper = f"""
{func_code}

def __wrapper():
    try:
        {func_name}({dummy_args})
    except:
        pass
"""

    return wrapper

In [10]:
# RUN DENGAN TIMEOUT (THREAD)
def run_with_timeout(code):

    result = {"status":None,"runtime":None,"message":""}

    def worker():

        try:

            local_env = {}

            dummy = io.StringIO()

            # override input agar tidak menunggu user
            local_env["input"] = lambda *args: "0"

            with contextlib.redirect_stdout(dummy), contextlib.redirect_stderr(dummy):

                exec(code, local_env)

                runtime = timeit.timeit(
                    "__wrapper()",
                    globals=local_env,
                    number=RUN_REPEAT
                )

            result["status"] = "SUCCESS"
            result["runtime"] = runtime / RUN_REPEAT

        except Exception as e:

            result["status"] = "FAILED"
            result["message"] = str(e)

    thread = threading.Thread(target=worker)

    thread.start()
    thread.join(TIMEOUT)

    if thread.is_alive():

        return "TIMEOUT", None, "Execution timeout"

    return result["status"], result["runtime"], result["message"]

In [11]:
# ANALISIS DATASET

records = []

print("\n===== MULAI ANALISIS RUNTIME =====\n")

skip_funcs = ["main","menu","run","start","program", "main_menu"]

for root, dirs, files in os.walk(dataset_folder):

    for file in files:

        if not file.endswith(".py"):
            continue

        file_path = os.path.join(root, file)

        nim, module, file_name = extract_metadata(file_path, dataset_folder)

        functions = extract_functions(file_path)

        if not functions:

            records.append({
                "timestamp": datetime.now(),
                "nim": nim,
                "module": module,
                "file_name": file_name,
                "function_name": "NO_FUNCTION",
                "status": "FAILED",
                "runtime_seconds": None,
                "message": "No function detected"
            })

            continue

        for func_name, func_code in functions:

            if func_name.lower() in skip_funcs:
                continue

            print(f"[RUN] {nim}/{module}/{file_name} → {func_name}")

            wrapper_code = generate_wrapper(func_name, func_code)

            status, runtime, message = run_with_timeout(wrapper_code)

            records.append({
                "timestamp": datetime.now(),
                "nim": nim,
                "module": module,
                "file_name": file_name,
                "function_name": func_name,
                "status": status,
                "runtime_seconds": runtime,
                "message": message
            })

print("\n===== ANALISIS SELESAI =====\n")


===== MULAI ANALISIS RUNTIME =====

[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → sigmoid
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → sigmoid_derivative
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → sigmoid
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → sigmoid_derivative
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → relu
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → relu_derivative
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → train_xor
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P2_JS13.py → build_model
[RUN] 2241720092/Klasterisasi/P2_JS04.py → find_clusters
[RUN] 2241720092/Klasterisasi/P2_JS04.py → plot_pixels
[RUN] 2341720081/JS04/P2_JS04.py → find_clusters
[RUN] 2341720081/

In [12]:
# Simpan
import sys
# kembalikan stdout normal
sys.stdout = sys.__stdout__
sys.stderr = sys.__stderr__

df = pd.DataFrame(records)

df["runtime_seconds"] = pd.to_numeric(df["runtime_seconds"], errors="coerce")

df.to_csv(output_csv, index=False)
df.to_excel(output_excel, index=False)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=4, default=str)

print("Hasil disimpan di:")
print(output_csv)
print(output_excel)
print(output_json)

# TAMPILKAN HASIL
print("\n===== HASIL RUNTIME PER FUNCTION =====\n")

display_columns = [
    "nim",
    "module",
    "file_name",
    "function_name",
    "status",
    "runtime_seconds"
]

display(df[display_columns])


,nim,module,file_name,function_name,status,runtime_seconds
0,2241720092,root,Uts.py,NO_FUNCTION,FAILED,NaN
1,2241720092,Artificial Neural Network (ANN) dan Evaluasi C...,P1_JS13.py,sigmoid,SUCCESS,4.233336e-06
2,2241720092,Artificial Neural Network (ANN) dan Evaluasi C...,P1_JS13.py,sigmoid_derivative,SUCCESS,8.999972e-07
3,2241720092,Artificial Neural Network (ANN) dan Evaluasi C...,P1_JS13.py,sigmoid,SUCCESS,3.033332e-06
4,2241720092,Artificial Neural Network (ANN) dan Evaluasi C...,P1_JS13.py,sigmoid_derivative,SUCCESS,1.099999e-06
...,...,...,...,...,...,...
553,2341720187,UAS,UAS.py,extract_hog_features,SUCCESS,1.666665e-06
554,2341720187,UAS,UAS.py,extract_lbp_features,SUCCESS,1.633331e-06
555,2341720187,UAS,UAS.py,extract_features,SUCCESS,1.833333e-06
556,2341720187,UAS,UAS.py,predict_and_show,SUCCESS,1.799999e-06


In [13]:
# RINGKASAN
print("\n===== RINGKASAN =====\n")

summary = {
    "Total function dianalisis": len(df),
    "SUCCESS": (df["status"] == "SUCCESS").sum(),
    "FAILED": (df["status"] == "FAILED").sum(),
    "TIMEOUT": (df["status"] == "TIMEOUT").sum(),
    "Rata-rata runtime": round(df["runtime_seconds"].mean(), 6)
}

summary_df = pd.DataFrame(list(summary.items()), columns=["Metric","Value"])

display(summary_df)

print("\nTop 5 runtime terlama:\n")

top_runtime = (
    df.sort_values("runtime_seconds", ascending=False)
    [["nim","module","file_name","function_name","runtime_seconds"]]
    .head(5)
)

display(top_runtime)

,Metric,Value
0,Total function dianalisis,558.000000
1,SUCCESS,268.000000
2,FAILED,290.000000
3,TIMEOUT,0.000000
4,Rata-rata runtime,0.000003


,nim,module,file_name,function_name,runtime_seconds
241,2341720096,JS07,P5_JS07.py,run_hnsw,0.000130
182,2341720095,JS11,P5_JS11.py,load_dataset,0.000018
498,2341720187,JS08,TP_JS08.py,count_outliers,0.000014
492,2341720187,JS07,P5_JS07.py,recall_at_k,0.000011
378,2341720117,PBL_FastAPI,Salinan PBL_ML Regression Risk_V1.py,tampilkan_hasil,0.000009
